# Lesson 02: The Attention Mechanism

Goal: understand Query/Key/Value and scaled dot-product attention with small, concrete examples.

## Learning goals

- Explain Q, K, V intuitively
- Compute scaled dot-product attention step by step
- Understand what softmax does in attention
- See how causal masking prevents looking ahead

## Intuition

For each token:
- **Query (Q):** what this token is looking for
- **Key (K):** what this token offers for matching
- **Value (V):** the information this token contributes

Attention scores are query-key matches. Softmax turns scores into weights.
Then we take a weighted sum of values.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

def softmax(x, axis=-1):
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x_shift)
    return e / np.sum(e, axis=axis, keepdims=True)

In [ ]:
# Toy token representations (3 tokens, d_model=4)
X = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 1.0],
])

d_model = X.shape[1]
d_k = 2

W_Q = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0], [0.0, 0.5]])
W_K = np.array([[1.0, 0.0], [0.0, 1.0], [0.2, 0.0], [0.0, 0.2]])
W_V = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]])

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print('Q:
', Q)
print('K:
', K)
print('V:
', V)

In [ ]:
# Scaled dot-product attention
scores = (Q @ K.T) / np.sqrt(d_k)
weights = softmax(scores, axis=-1)
output = weights @ V

print('Scores:
', scores)
print('Weights (rows sum to 1):
', weights)
print('Attention output:
', output)

Formula:

`Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V`

In [ ]:
# Causal mask: token i can only attend to positions <= i
n = scores.shape[0]
mask = np.triu(np.ones((n, n), dtype=bool), k=1)
masked_scores = scores.copy()
masked_scores[mask] = -1e9

masked_weights = softmax(masked_scores, axis=-1)
masked_output = masked_weights @ V

print('Causal mask (True means blocked):
', mask)
print('Masked weights:
', masked_weights)
print('Masked output:
', masked_output)

## Checkpoints

1. Why do we divide by `sqrt(d_k)`?
2. What happens if we remove softmax?
3. Why is causal masking essential for next-token prediction?

## Next lesson

`03_transformer_block.ipynb`: how attention fits inside a full Transformer block (residuals, layer norm, MLP).